# Capstone Project: Data Science AI Tutor

**Title:** Capstone Project: Data Science AI Tutor  
**Difficulty:** Expert  
**Notebook:** 07 of 07  

---

> *This final notebook brings together every concept from Notebooks 01-06.*

You will build a **Data Science AI Tutor** -- an intelligent assistant that can:

- Answer Data Science questions using RAG
- Explain concepts at beginner, intermediate, or advanced levels
- Generate Python code examples
- Create quizzes to test understanding
- Perform numerical calculations using tools
- Return structured, parseable output
- Show sources for its answers

## Learning Objectives

By the end of this capstone you will be able to:

1. **Integrate** RAG, tools, chains, and structured output
2. **Build** a multi-component AI system incrementally
3. **Configure** both API and local Ollama implementations
4. **Implement** RAG with source display
5. **Create** custom tools for autonomous use
6. **Use** Pydantic models for structured output
7. **Design** a complete AI application architecture
8. **Evaluate** your system and identify improvements

## Prerequisites

| Concept | Source |
|---------|--------|
| Chat models, prompts, messages | Notebook 02 |
| LCEL chains | Notebook 03 |
| Embeddings and vector stores | Notebook 04 |
| RAG pipelines | Notebook 05 |
| Tools and agents | Notebook 06 |

> **Time estimate:** 120-180 minutes

## Architecture Overview

Here is what we are building:

```mermaid
graph TD
    A[Student] --> B[Data Science AI Tutor]
    B --> C[LLM]
    B --> D[Prompt System]
    B --> E[RAG Retriever]
    E --> F[Vector Store]
    E --> G[Knowledge Base]
    B --> H[Tools]
    H --> I[NumPy/Pandas]
    B --> J[Structured Output]
    style A fill:#e1f5fe
    style B fill:#fff3e0
    style C fill:#e8f5e9
    style E fill:#f3e5f5
    style H fill:#fce4ec
```

We will build this **incrementally** in 12 stages.

## Setup

In [ ]:
import os
import json
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from typing import Literal

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.runnables import RunnablePassthrough
from langchain_core.tools import tool

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_ollama import ChatOllama, OllamaEmbeddings

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from pydantic import BaseModel, Field

load_dotenv()
print('All imports loaded successfully!')

In [ ]:
api_key = os.getenv('OPENAI_API_KEY', '')
print(f'API key: {"set" if api_key else "not found -- use Ollama"}')

---

## Stage 1: Configuration

In [ ]:
class Config:
    OPENAI_MODEL = 'gpt-4o-mini'
    OLLAMA_MODEL = 'llama3.2'
    TEMPERATURE = 0.3
    OPENAI_EMBEDDING_MODEL = 'text-embedding-3-small'
    OLLAMA_EMBEDDING_MODEL = 'nomic-embed-text'
    CHUNK_SIZE = 500
    CHUNK_OVERLAP = 100
    RETRIEVER_TOP_K = 3
    KNOWLEDGE_DIR = '../data/ds_notes'
    PERSIST_DIR = '../data/chroma_db'
    USE_OLLAMA = False

print(f'Mode: {"Ollama" if Config.USE_OLLAMA else "OpenAI API"}')

---

## Stage 2: LLM

In [ ]:
def create_llm():
    if Config.USE_OLLAMA:
        return ChatOllama(model=Config.OLLAMA_MODEL, temperature=Config.TEMPERATURE)
    return ChatOpenAI(model=Config.OPENAI_MODEL, temperature=Config.TEMPERATURE)

llm = create_llm()
print(f'LLM: {type(llm).__name__}')

resp = llm.invoke([HumanMessage(content='Say hello.')])
print(f'Test: {resp.content}')

---

## Stage 3: Knowledge Base

In [ ]:
from pathlib import Path

def create_knowledge_base():
    kb_dir = Path(Config.KNOWLEDGE_DIR)
    kb_dir.mkdir(parents=True, exist_ok=True)

    kb = {}
    kb['python.md'] = '# Python for Data Science\n\nPython is the most popular language for data science. Key libraries: NumPy, Pandas, Matplotlib, Scikit-learn.\n\n## Key Operations\n- Slicing and indexing\n- Vectorized operations\n- List comprehensions'
    kb['pandas.md'] = '# Pandas for Data Science\n\nPandas provides Series and DataFrame data structures.\n\n## Core Operations\n- pd.read_csv(): Load CSV\n- df.head(): Preview\n- df.describe(): Statistics\n- df.groupby(): Group and aggregate'
    kb['machine_learning.md'] = '# ML Fundamentals\n\n## Supervised Learning\n- Classification, Regression\n- Algorithms: Linear Regression, Decision Trees, Random Forest, SVM\n\n## Unsupervised\n- Clustering: K-Means, DBSCAN\n- Dimensionality Reduction: PCA\n\n## Evaluation\n- Accuracy, Precision, Recall, F1, AUC-ROC'
    kb['statistics.md'] = '# Statistics for Data Science\n\n## Descriptive\n- Mean, Median, Mode\n- Standard Deviation, Variance\n\n## Probability\n- Distributions: Normal, Binomial\n- Central Limit Theorem\n- Bayes Theorem\n\n## Hypothesis Testing\n- p-values, significance levels'
    kb['evaluation.md'] = '# Model Evaluation\n\n## Classification\n- Accuracy: (TP+TN)/Total\n- Precision: TP/(TP+FP)\n- Recall: TP/(TP+FN)\n- F1: Harmonic mean of precision and recall\n\n## Regression\n- MSE, RMSE, MAE, R-squared\n\n## Imbalanced Classes\n- Use F1 or AUC-ROC instead of accuracy'
    for fn, text in kb.items():
        fp = kb_dir / fn
        if not fp.exists(): fp.write_text(text)

    docs = [Document(page_content=(kb_dir/f).read_text(),
        metadata={'source': f, 'topic': f.replace('.md','')}) for f in kb_dir.glob('*.md')]
    print(f'Loaded {len(docs)} documents')
    return docs

docs = create_knowledge_base()

---

## Stage 4: Embeddings

In [ ]:
def create_embeddings():
    if Config.USE_OLLAMA: return OllamaEmbeddings(model=Config.OLLAMA_EMBEDDING_MODEL)
    return OpenAIEmbeddings(model=Config.OPENAI_EMBEDDING_MODEL)

embeddings = create_embeddings()
print(f'Embeddings: {type(embeddings).__name__}')

---

## Stage 5: Vector Store

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=Config.CHUNK_SIZE, chunk_overlap=Config.CHUNK_OVERLAP)
chunks = splitter.split_documents(docs)
print(f'{len(docs)} docs -> {len(chunks)} chunks')

In [ ]:
import shutil
pd2 = Path(Config.PERSIST_DIR)
if pd2.exists(): shutil.rmtree(pd2)
pd2.mkdir(parents=True, exist_ok=True)
vectorstore = Chroma.from_documents(chunks, embeddings, persist_directory=str(pd2))
print(f'Vector store: {vectorstore._collection.count()} chunks')

---

## Stage 6: Retriever

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={'k': Config.RETRIEVER_TOP_K})

for q in ['What is F1 score?', 'How to handle imbalanced classes?']:
    docs = retriever.invoke(q)
    print(f'Q: {q}')
    for i, d in enumerate(docs):
        print(f'  {i+1}. [{d.metadata["topic"]}] {d.page_content[:50]}...')
    print()

---

## Stage 7: RAG

In [ ]:
def format_docs(docs):
    return '\n---\n'.join(f'[{d.metadata.get("source","?")}]\n{d.page_content}' for d in docs)

rag_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are an expert Data Science tutor. Use context to answer.'),
    ('human', 'Context:\n{context}\n\nQ: {question}\n\nA:')])

rag_chain = ({'context': retriever | format_docs, 'question': RunnablePassthrough()}
    | rag_prompt | llm | StrOutputParser())

print('RAG chain built!')

In [ ]:
for q in ['What metric for imbalanced classes?', 'Precision vs recall?']:
    print(f'Q: {q}')
    print(f'A: {rag_chain.invoke(q)[:200]}...\n')

---

## Stage 8: Tools

In [ ]:
@tool
def calculate_mean(values: list[float]) -> str:
    '''Calculate the arithmetic mean.'''
    return f'Mean = {np.mean(values):.4f}' if values else 'Error: empty'

@tool
def calculate_std(values: list[float]) -> str:
    '''Calculate the standard deviation.'''
    return f'Std = {np.std(values, ddof=1):.4f}' if len(values) > 1 else 'Error: need 2+'

@tool
def calculate_correlation(x_values: list[float], y_values: list[float]) -> str:
    '''Calculate Pearson correlation.'''
    c = np.corrcoef(x_values, y_values)[0, 1]
    return f'Correlation = {c:.4f} ({"strong" if abs(c)>0.7 else "moderate" if abs(c)>0.4 else "weak"})'

@tool
def dataset_summary(csv_content: str) -> str:
    '''Generate descriptive statistics from CSV.'''
    from io import StringIO
    df = pd.read_csv(StringIO(csv_content))
    return f'Shape: {df.shape}\n{df.describe().to_string()}'

tools = [calculate_mean, calculate_std, calculate_correlation, dataset_summary]
print(f'Created {len(tools)} tools')

---

## Stage 9: Structured Output

In [ ]:
class ConceptExplanation(BaseModel):
    concept: str; level: Literal['beginner','intermediate','advanced']
    definition: str; intuition: str; example: str
    common_mistakes: list[str]; related_concepts: list[str]

class QuizQuestion(BaseModel):
    question: str; options: list[str]; correct_answer: str
    explanation: str; difficulty: Literal['easy','medium','hard']

explanation_chain = (
    ChatPromptTemplate.from_messages([('system','Generate structured DS explanation.'),
    ('human','Explain {concept} at {level} level.')])
    | llm.with_structured_output(ConceptExplanation))

print('Structured output chain built!')

In [ ]:
result = explanation_chain.invoke({'concept': 'Random Forest', 'level': 'intermediate'})
print(f'Concept: {result.concept}')
print(f'Definition: {result.definition}')
print(f'Intuition: {result.intuition}')

---

## Stage 10: The Complete Tutor

In [ ]:
class DataScienceTutor:
    def __init__(self, llm, retriever, tools):
        self.llm = llm; self.retriever = retriever; self.tools = tools
        self._setup_chains()

    def _setup_chains(self):
        rp = ChatPromptTemplate.from_messages([
            ('system','Use context to answer. Be educational.'),
            ('human','Context:\n{context}\n\nQ: {question}\n\nA:')])
        self.rag_chain = ({'context': self.retriever | format_docs, 'question': RunnablePassthrough()}
            | rp | self.llm | StrOutputParser())
        self.explanation_chain = (ChatPromptTemplate.from_messages([
            ('system','Generate structured DS explanation.'),
            ('human','Explain {concept} at {level} level.')])
            | self.llm.with_structured_output(ConceptExplanation))
        self.quiz_chain = (ChatPromptTemplate.from_messages([
            ('system','Generate a quiz question.'),
            ('human','{difficulty} quiz about: {topic}')])
            | self.llm.with_structured_output(QuizQuestion))

    def ask(self, q):
        a = self.rag_chain.invoke(q); d = self.retriever.invoke(q)
        return {'answer': a, 'sources': list(set(x.metadata.get('source','?') for x in d))}

    def explain(self, concept, level='intermediate'):
        return self.explanation_chain.invoke({'concept': concept, 'level': level})

    def quiz(self, topic, difficulty='medium'):
        return self.quiz_chain.invoke({'topic': topic, 'difficulty': difficulty})

    def calculate(self, name, **kw):
        m = {t.name: t for t in self.tools}
        return m.get(name, lambda x: 'Unknown tool').invoke(kw)

tutor = DataScienceTutor(llm, retriever, tools)
print('Data Science AI Tutor is ready!')

---

## Stage 11: Testing

In [ ]:
print('='*60 + '\nTEST 1: RAG Q&A')
r = tutor.ask('What is cross-validation?')
print(f'Answer: {r["answer"][:300]}...')
print(f'Sources: {r["sources"]}')

In [ ]:
print('='*60 + '\nTEST 2: Explanation')
e = tutor.explain('Gradient Descent', 'beginner')
print(f'{e.concept} ({e.level}): {e.definition}')

In [ ]:
print('='*60 + '\nTEST 3: Quiz')
q = tutor.quiz('ML metrics', 'medium')
print(f'Q: {q.question}')
for i, o in enumerate(q.options):
    print(f'  {chr(65+i)}. {o}' + (' <--' if o == q.correct_answer else ''))

In [ ]:
print('='*60 + '\nTEST 4: Tools')
print(tutor.calculate('calculate_mean', values=[23.5, 45.2, 67.8, 12.1, 89.4]))
print(tutor.calculate('calculate_correlation',
    x_values=[72,85,78,90,88,95,92,97,75,82],
    y_values=[8,12,10,15,14,18,16,20,9,11]))

In [ ]:
print('='*60 + '\nTEST 5: Dataset')
df = pd.DataFrame({'hours': [2,4,6,8,10], 'score': [45,55,65,75,85]})
print(tutor.calculate('dataset_summary', csv_content=df.to_csv(index=False)))

---

## Stage 12: Possible Improvements

| Improvement | Description | Difficulty |
|-------------|-------------|------------|
| **Chat Memory** | Remember previous questions | Medium |
| **Better Chunking** | Experiment with chunk sizes | Easy |
| **Reranking** | Better retrieval ordering | Medium |
| **Web Search** | Add web search tool | Medium |
| **Streaming** | Stream responses | Easy |
| **Multi-modal** | Accept images and files | Hard |
| **Evaluation** | Automated quality checks | Hard |
| **Deployment** | Streamlit or FastAPI | Medium |
| **Guardrails** | Content filtering and safety | Hard |

---

## API vs Local Ollama

| Aspect | OpenAI API | Local Ollama |
|--------|-----------|--------------|
| **Setup** | pip install | Install Ollama + pull models |
| **Internet** | Required | Not required |
| **Cost** | Per-token | Free after download |
| **Privacy** | Sent to OpenAI | Stays local |
| **Quality** | GPT-4o: excellent | Llama 3.2: good |
| **Speed** | Network dependent | Hardware dependent |

Switch by setting `Config.USE_OLLAMA = True`.

---

## Final Architecture

```mermaid
graph TD
    A[Student] --> B[Tutor]
    B --> C{Router}
    C -->|Question| D[RAG]
    C -->|Explain| E[Structured]
    C -->|Quiz| F[Quiz]
    C -->|Calculate| G[Tools]
    D --> P[LLM]
    E --> P
    F --> P
    P --> Q{API or Local}
    Q -->|API| R[OpenAI]
    Q -->|Local| S[Ollama]
    style P fill:#f3e5f5
```

---

## Where to Go Next

### LangGraph
Graph-based execution with branching, loops, and human-in-the-loop.
```bash
pip install langgraph
```

### Advanced RAG
Query rewriting, hybrid search, reranking, multi-query RAG, CRAG.

### RAG Evaluation
Context relevance, faithfulness, answer correctness. Tools: RAGAS, LangSmith.

### Memory and State
BufferMemory, SummaryMemory, LangGraph checkpointing.

### Observability
LangSmith for tracing, debugging, cost monitoring.

### Production
FastAPI, Streamlit/Gradio, Docker, Cloud deployment.

### Guardrails
Input validation, output filtering, hallucination detection.

### MCP
Model Context Protocol -- universal tool interface.

### Multimodal
Vision, Audio, Documents, Code execution.

---

## Capstone Assignment: Build Your Own Domain-Specific AI Tutor

### Requirements

1. **Knowledge Base** (min 5 documents, 500+ words each)
2. **RAG System** -- loading, splitting, embedding, retrieval
3. **Custom Tools** (min 3 domain-specific tools)
4. **Structured Output** (min 2 Pydantic models)
5. **Complete Tutor** class with ask(), explain(), calculate()
6. **Documentation** -- architecture diagram, examples, limitations

### Domain Ideas

| Domain | Example Tools |
|--------|---------------|
| Biology | Population growth calculator |
| Economics | GDP, elasticity calculator |
| Chemistry | Molar mass, pH calculator |
| Finance | ROI, Sharpe ratio calculator |
| Music Theory | Interval, chord analyzer |
| Cooking | Unit converter, scaling calculator |

### Extension Ideas

- **Streamlit web interface**
- **Chat memory** for multi-turn conversations
- **Image analysis** for charts
- **Evaluation suite** for quality
- **Cloud deployment**

---

## Key Takeaways

| Component | Technology | Purpose |
|-----------|-----------|---------|
| LLM | ChatOpenAI / ChatOllama | Language generation |
| RAG | Retriever + Chroma + LLM | Knowledge-grounded answers |
| Tools | @tool + NumPy/Pandas | Numerical calculations |
| Structured Output | Pydantic | Typed responses |
| Embeddings | OpenAI / Ollama | Semantic search |
| Text Splitting | RecursiveCharacterTextSplitter | Chunking |
| Vector Store | Chroma | Persistent storage |

```
Notebooks 01-06: Foundations
  01. Introduction      -- What is LangChain?
  02. Models & Prompts  -- Chat models, messages, templates
  03. LCEL & Chains     -- Pipeline composition
  04. Embeddings        -- Vector representations
  05. RAG               -- Retrieval-augmented generation
  06. Tools & Agents    -- Dynamic tool calling

Notebook 07: Capstone
  07. AI Tutor          -- Everything combined!
```

> *The best way to learn LangChain is to build something real.*